# FR→EN Transformer: Colab training
Runtime → Change runtime type → **T4 GPU**. Add `HF_TOKEN` and `WANDB_API_KEY` in Colab **Secrets** (key icon). Checkpoints go to Google Drive, so after a disconnect just **run all cells again**: training resumes from `last.pt`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
OUT = '/content/drive/MyDrive/fr-en-transformer/base'
WORK_CACHE = '/content/drive/MyDrive/fr-en-transformer/work'

In [ ]:
%cd /content
!test -d transfromer-model || git clone https://github.com/bad2212/transfromer-model.git
%cd /content/transfromer-model
!git pull -q
!pip -q install sentencepiece wandb datasets safetensors pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# data + tokenizer: build once, cache on Drive (keeps the same spm.model across sessions)
!mkdir -p $WORK_CACHE
!test -f $WORK_CACHE/spm.model || (python -m src.prepare_data --config configs/base.yaml && cp work/* $WORK_CACHE/)
!mkdir -p work && cp -n $WORK_CACHE/* work/
!cat work/data_stats.json

In [ ]:
# throughput check (~3 min): look at tgt_tokens_per_s; < ~12k => switch to configs/base_6x3.yaml
!python -m src.train --config configs/base.yaml --out_dir /content/throughput_test --max_steps 300 2>&1 | grep -E 'params|step [0-9]+00 '

In [ ]:
# main run (resumable). Re-run this cell after any disconnect.
!python tests/test_sanity.py && python -m src.train --config configs/base.yaml --out_dir $OUT

In [ ]:
# checkpoint averaging + dev decoding sweep + test predictions
!python -m src.predict --config configs/base.yaml --out_dir $OUT --avg 5 --sweep
!python score.py --gold data/dev/labels.jsonl --pred $OUT/eval/dev_predictions.json